# APTOS 2019 — Diabetic Retinopathy Severity Grading
## ResNet50 | 5-class classification (0–4) | Kaggle-ready

This notebook trains **only the DR-severity branch**:

- **Dataset:** APTOS 2019 Blindness Detection
- **Task:** Diabetic-retinopathy severity grading
- **Output:** `0 = No DR`, `1 = Mild`, `2 = Moderate`, `3 = Severe`, `4 = Proliferative DR`
- **Backbone:** ImageNet-pretrained ResNet50
- **Independent from:** IDRiD macular-edema-risk training

### Measures included against common training problems

- Stratified train/validation split
- Fixed seeds and reproducibility controls
- Balanced class weights
- Fundus dark-border cropping
- Conservative augmentation
- ImageNet transfer learning
- Two-stage training: frozen head → partial backbone fine-tuning
- AdamW + weight decay
- Label smoothing
- Gradient clipping
- Mixed precision on GPU
- ReduceLROnPlateau
- Best-model checkpointing
- Accuracy + Macro-F1 + Quadratic Weighted Kappa
- Confusion matrix and per-class report
- Final `.keras` model and `.weights.h5` export
- Optional Kaggle submission generation

> For DR grading, do not judge the model from accuracy alone. Macro-F1, per-class recall and QWK are also important because the dataset is imbalanced and the labels are ordinal.

### Model-specific configuration
- Input size: **384×384**
- Batch size: **16**
- Fine-tuning: last **80** backbone layers considered trainable, with BatchNorm layers kept frozen.


### Fixed 150-epoch training schedule
- **Stage 1:** 10 epochs with the pretrained backbone frozen
- **Stage 2:** 140 epochs of partial backbone fine-tuning
- **Total:** 150 epochs
- Fixed 150-epoch training is intentionally disabled so all models complete the same 150-epoch schedule.
- The best model is still selected by validation QWK using `ModelCheckpoint`.
- `ReduceLROnPlateau` remains enabled to lower the learning rate when validation progress stalls.


In [ ]:
# ============================================================
# 1. Imports, reproducibility, GPU and mixed precision
# ============================================================

import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    cohen_kappa_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU(s):", gpus)

USE_MIXED_PRECISION = len(gpus) > 0
if USE_MIXED_PRECISION:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision policy:", mixed_precision.global_policy())
else:
    print("GPU not detected. Training will work but will be much slower.")

In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

IMG_SIZE = 384
BATCH_SIZE = 16

HEAD_EPOCHS = 10
HEAD_LR = 3e-4

FINETUNE_EPOCHS = 140
FINETUNE_LR = 2e-5
UNFREEZE_LAST_N = 80

VAL_SIZE = 0.20
LABEL_SMOOTHING = 0.05
WEIGHT_DECAY = 1e-4
GRAD_CLIPNORM = 1.0

NUM_CLASSES = 5
CLASS_NAMES = {
    0: "No DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative DR",
}

AUTOTUNE = tf.data.AUTOTUNE
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = WORK_DIR / "aptos_resnet50_dr_grading_best.keras"
FINAL_MODEL_PATH = WORK_DIR / "aptos_resnet50_dr_grading_final.keras"
WEIGHTS_PATH = WORK_DIR / "aptos_resnet50_dr_grading.weights.h5"

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Outputs:", WORK_DIR)

In [ ]:
# ============================================================
# 3. Automatically locate APTOS files
# ============================================================

def locate_aptos():
    input_root = Path("/kaggle/input")
    csv_candidates = list(input_root.rglob("train.csv"))

    valid = []
    for csv_path in csv_candidates:
        try:
            preview = pd.read_csv(csv_path, nrows=5)
            if {"id_code", "diagnosis"}.issubset(preview.columns):
                valid.append(csv_path)
        except Exception:
            pass

    if not valid:
        raise FileNotFoundError(
            "Could not find APTOS train.csv with columns 'id_code' and 'diagnosis'. "
            "Attach the APTOS 2019 competition data to this Kaggle notebook."
        )

    valid = sorted(
        valid,
        key=lambda p: (
            "aptos2019-blindness-detection" not in str(p).lower(),
            len(str(p))
        )
    )
    train_csv = valid[0]
    root = train_csv.parent

    train_candidates = [
        root / "train_images",
        root / "train",
        root / "images" / "train",
    ]
    train_dir = next((p for p in train_candidates if p.exists()), None)

    if train_dir is None:
        possible = [p for p in root.rglob("*") if p.is_dir() and p.name == "train_images"]
        if possible:
            train_dir = possible[0]

    if train_dir is None:
        raise FileNotFoundError(f"Could not locate train_images near {train_csv}")

    return (
        train_csv,
        train_dir,
        root / "test.csv",
        root / "test_images",
        root / "sample_submission.csv",
    )

TRAIN_CSV, TRAIN_IMG_DIR, TEST_CSV, TEST_IMG_DIR, SAMPLE_SUBMISSION = locate_aptos()

print("TRAIN_CSV:", TRAIN_CSV)
print("TRAIN_IMG_DIR:", TRAIN_IMG_DIR)
print("TEST_CSV exists:", TEST_CSV.exists())
print("TEST_IMG_DIR exists:", TEST_IMG_DIR.exists())

In [ ]:
# ============================================================
# 4. Read labels and verify files
# ============================================================

df = pd.read_csv(TRAIN_CSV)
df = df[["id_code", "diagnosis"]].copy()
df["diagnosis"] = df["diagnosis"].astype(int)

def resolve_image_path(image_id, image_dir):
    for ext in (".png", ".jpg", ".jpeg"):
        p = image_dir / f"{image_id}{ext}"
        if p.exists():
            return str(p)
    return None

df["image_path"] = df["id_code"].apply(lambda x: resolve_image_path(x, TRAIN_IMG_DIR))

missing = int(df["image_path"].isna().sum())
print("Rows:", len(df))
print("Missing image files:", missing)

if missing:
    df = df.dropna(subset=["image_path"]).reset_index(drop=True)

assert set(df["diagnosis"].unique()).issubset(set(range(5))), "Unexpected labels detected."

class_table = pd.DataFrame({
    "class": list(range(NUM_CLASSES)),
    "name": [CLASS_NAMES[i] for i in range(NUM_CLASSES)],
    "count": [int((df["diagnosis"] == i).sum()) for i in range(NUM_CLASSES)],
})
display(class_table)

In [ ]:
# ============================================================
# 5. Class distribution
# ============================================================

counts = df["diagnosis"].value_counts().sort_index()

plt.figure(figsize=(8, 4))
plt.bar([f"{i}: {CLASS_NAMES[i]}" for i in counts.index], counts.values)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Images")
plt.title("APTOS DR grade distribution")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6. Stratified train/validation split + class weights
# ============================================================

train_df, val_df = train_test_split(
    df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=df["diagnosis"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))

print("\nTrain class fractions:")
print(train_df["diagnosis"].value_counts(normalize=True).sort_index().round(4))

print("\nValidation class fractions:")
print(val_df["diagnosis"].value_counts(normalize=True).sort_index().round(4))

classes = np.arange(NUM_CLASSES)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["diagnosis"].values,
)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

print("\nClass weights:")
print(class_weight)

In [ ]:
# ============================================================
# 7. Image preprocessing
#    Removes large dark borders, resizes and preserves RGB.
# ============================================================

def crop_dark_border(image, threshold=10.0):
    gray = tf.reduce_mean(image, axis=-1)
    mask = gray > threshold
    coords = tf.where(mask)

    def cropped():
        y_min = tf.reduce_min(coords[:, 0])
        y_max = tf.reduce_max(coords[:, 0])
        x_min = tf.reduce_min(coords[:, 1])
        x_max = tf.reduce_max(coords[:, 1])
        return image[y_min:y_max + 1, x_min:x_max + 1, :]

    return tf.cond(tf.shape(coords)[0] > 0, cropped, lambda: image)


def load_image(path, label=None):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.cast(image, tf.float32)

    image = crop_dark_border(image)
    image = tf.image.resize_with_pad(
        image,
        target_height=IMG_SIZE,
        target_width=IMG_SIZE,
        method="bilinear",
        antialias=True,
    )
    image.set_shape([IMG_SIZE, IMG_SIZE, 3])

    if label is None:
        return image

    label = tf.one_hot(tf.cast(label, tf.int32), depth=NUM_CLASSES)
    return image, label

In [ ]:
# ============================================================
# 8. Conservative augmentation + tf.data
# ============================================================

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.06, fill_mode="reflect"),
        layers.RandomZoom(
            height_factor=(-0.08, 0.08),
            width_factor=(-0.08, 0.08),
            fill_mode="reflect",
        ),
        layers.RandomTranslation(
            height_factor=0.04,
            width_factor=0.04,
            fill_mode="reflect",
        ),
        layers.RandomContrast(0.12),
    ],
    name="fundus_augmentation",
)

def make_dataset(frame, training=False):
    paths = frame["image_path"].astype(str).values
    labels_np = frame["diagnosis"].astype(np.int32).values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels_np))

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE,
        )

    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)

print(train_ds)
print(val_ds)

In [ ]:
# ============================================================
# 9. Sanity-check augmented images
# ============================================================

images, labels_batch = next(iter(train_ds.take(1)))

plt.figure(figsize=(12, 8))
for i in range(min(12, int(images.shape[0]))):
    ax = plt.subplot(3, 4, i + 1)
    img = tf.cast(tf.clip_by_value(images[i], 0, 255), tf.uint8)
    plt.imshow(img)
    grade = int(tf.argmax(labels_batch[i]).numpy())
    plt.title(f"{grade}: {CLASS_NAMES[grade]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 10. Build ResNet50 classifier
# ============================================================

def build_model():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="fundus_image")

    # Keep ResNet50 preprocessing inside the model so the exported
    # .keras model accepts ordinary RGB values in the range 0..255.
    x = tf.keras.applications.resnet50.preprocess_input(inputs)

    try:
        backbone = tf.keras.applications.ResNet50(
            include_top=False,
            weights="imagenet",
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
        )
    except Exception as e:
        raise RuntimeError(
            "ImageNet weights could not be loaded. Enable Internet on Kaggle "
            "for the first run or attach compatible ResNet50 ImageNet weights. "
            "Random initialization is intentionally not used because it usually "
            "reduces performance for this dataset."
        ) from e

    backbone.trainable = False

    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pool")(x)
    x = layers.BatchNormalization(name="head_bn")(x)
    x = layers.Dropout(0.40, name="dropout_1")(x)
    x = layers.Dense(
        256,
        activation="swish",
        kernel_regularizer=regularizers.l2(1e-4),
        name="head_dense",
    )(x)
    x = layers.Dropout(0.30, name="dropout_2")(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="dr_grade",
    )(x)

    model = keras.Model(inputs, outputs, name="APTOS_ResNet50_DR_Grader")
    return model, backbone

model, backbone = build_model()
model.summary()

In [ ]:
# ============================================================
# 11. Compile helper
# ============================================================

def make_optimizer(lr):
    return tf.keras.optimizers.AdamW(
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        clipnorm=GRAD_CLIPNORM,
    )

loss_fn = tf.keras.losses.CategoricalCrossentropy(
    label_smoothing=LABEL_SMOOTHING
)

def compile_model(model_obj, lr):
    model_obj.compile(
        optimizer=make_optimizer(lr),
        loss=loss_fn,
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
            tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top2_accuracy"),
        ],
    )

compile_model(model, HEAD_LR)

In [ ]:
# ============================================================
# 12. QWK and Macro-F1 callback
# ============================================================

class ValidationMetrics(keras.callbacks.Callback):
    def __init__(self, val_dataset, y_true):
        super().__init__()
        self.val_dataset = val_dataset
        self.y_true = np.asarray(y_true, dtype=int)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        probs = self.model.predict(self.val_dataset, verbose=0)
        pred = np.argmax(probs, axis=1)

        acc = accuracy_score(self.y_true, pred)
        p, r, f1, _ = precision_recall_fscore_support(
            self.y_true,
            pred,
            average="macro",
            zero_division=0,
        )
        qwk = cohen_kappa_score(
            self.y_true,
            pred,
            weights="quadratic",
            labels=list(range(NUM_CLASSES)),
        )

        logs["val_macro_precision"] = float(p)
        logs["val_macro_recall"] = float(r)
        logs["val_macro_f1"] = float(f1)
        logs["val_qwk"] = float(qwk)

        print(
            f" — val_acc: {acc:.4f}"
            f" — val_macro_f1: {f1:.4f}"
            f" — val_qwk: {qwk:.4f}"
        )

In [ ]:
# ============================================================
# 13. Stage 1 — train classification head
# ============================================================

stage1_callbacks = [
    ValidationMetrics(val_ds, val_df["diagnosis"].values),
    keras.callbacks.ModelCheckpoint(
        filepath=str(WORK_DIR / "aptos_resnet50_stage1_best.keras"),
        monitor="val_qwk",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.35,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.TerminateOnNaN(),
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    class_weight=class_weight,
    callbacks=stage1_callbacks,
    verbose=1,
)

In [ ]:
# ============================================================
# 14. Stage 2 — partial ResNet50 fine-tuning
# ============================================================

backbone.trainable = True

for layer in backbone.layers[:-UNFREEZE_LAST_N]:
    layer.trainable = False

# BatchNorm stays frozen during fine-tuning for stability on a small dataset.
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(int(layer.trainable) for layer in backbone.layers)
print("Backbone layers:", len(backbone.layers))
print("Trainable backbone layers:", trainable_count)

compile_model(model, FINETUNE_LR)

stage2_callbacks = [
    ValidationMetrics(val_ds, val_df["diagnosis"].values),
    keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_qwk",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.30,
        patience=3,
        min_lr=5e-7,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(
        str(WORK_DIR / "aptos_resnet50_stage2_training_log.csv")
    ),
    keras.callbacks.TerminateOnNaN(),
]

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    class_weight=class_weight,
    callbacks=stage2_callbacks,
    verbose=1,
)

In [ ]:
# ============================================================
# 15. Merge and plot training history
# ============================================================

def combine_history(h1, h2):
    result = {}
    keys = set(h1.history.keys()) | set(h2.history.keys())
    for key in keys:
        result[key] = h1.history.get(key, []) + h2.history.get(key, [])
    return result

hist = combine_history(history_head, history_ft)

history_df = pd.DataFrame({k: pd.Series(v) for k, v in hist.items()})
history_df.to_csv(WORK_DIR / "aptos_resnet50_full_history.csv", index=False)

plt.figure(figsize=(9, 4))
if "loss" in hist:
    plt.plot(hist["loss"], label="train_loss")
if "val_loss" in hist:
    plt.plot(hist["val_loss"], label="val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
if "accuracy" in hist:
    plt.plot(hist["accuracy"], label="train_accuracy")
if "val_accuracy" in hist:
    plt.plot(hist["val_accuracy"], label="val_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training / Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
if "val_qwk" in hist:
    plt.plot(hist["val_qwk"], label="val_QWK")
if "val_macro_f1" in hist:
    plt.plot(hist["val_macro_f1"], label="val_macro_F1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Validation QWK and Macro-F1")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 16. Load best checkpoint and evaluate thoroughly
# ============================================================

if BEST_MODEL_PATH.exists():
    best_model = tf.keras.models.load_model(BEST_MODEL_PATH, compile=False)
    print("Loaded:", BEST_MODEL_PATH)
else:
    best_model = model
    print("Best checkpoint not found; evaluating current model.")

val_probs = best_model.predict(val_ds, verbose=1)
val_pred = np.argmax(val_probs, axis=1)
val_true = val_df["diagnosis"].values.astype(int)

accuracy = accuracy_score(val_true, val_pred)
precision, recall, macro_f1, _ = precision_recall_fscore_support(
    val_true,
    val_pred,
    average="macro",
    zero_division=0,
)
qwk = cohen_kappa_score(
    val_true,
    val_pred,
    weights="quadratic",
    labels=list(range(NUM_CLASSES)),
)

print(f"Validation Accuracy        : {accuracy:.4f}")
print(f"Validation Macro Precision : {precision:.4f}")
print(f"Validation Macro Recall    : {recall:.4f}")
print(f"Validation Macro F1        : {macro_f1:.4f}")
print(f"Validation QWK             : {qwk:.4f}")

report = classification_report(
    val_true,
    val_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=[CLASS_NAMES[i] for i in range(NUM_CLASSES)],
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).T
display(report_df)
report_df.to_csv(WORK_DIR / "aptos_resnet50_classification_report.csv")

metrics_dict = {
    "validation_accuracy": float(accuracy),
    "validation_macro_precision": float(precision),
    "validation_macro_recall": float(recall),
    "validation_macro_f1": float(macro_f1),
    "validation_qwk": float(qwk),
}
with open(WORK_DIR / "aptos_resnet50_metrics.json", "w") as f:
    json.dump(metrics_dict, f, indent=2)

In [ ]:
# ============================================================
# 17. Confusion matrix
# ============================================================

cm = confusion_matrix(
    val_true,
    val_pred,
    labels=list(range(NUM_CLASSES)),
)

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[f"{i}\n{CLASS_NAMES[i]}" for i in range(NUM_CLASSES)],
)
disp.plot(ax=ax, values_format="d", xticks_rotation=30)
plt.title("APTOS ResNet50 — Validation Confusion Matrix")
plt.tight_layout()
fig.savefig(
    WORK_DIR / "aptos_resnet50_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# 18. Example validation predictions
# ============================================================

val_paths = val_df["image_path"].values

plt.figure(figsize=(15, 12))
n_show = min(20, len(val_df))

for i in range(n_show):
    image = load_image(tf.constant(val_paths[i]))
    probs = best_model(tf.expand_dims(image, 0), training=False).numpy()[0]
    pred = int(np.argmax(probs))
    true = int(val_true[i])
    conf = float(np.max(probs))

    ax = plt.subplot(4, 5, i + 1)
    plt.imshow(tf.cast(tf.clip_by_value(image, 0, 255), tf.uint8))
    plt.title(f"T:{true}  P:{pred}\nConf:{conf:.2f}", fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 19. Export deployable model, weights and label map
# ============================================================

best_model.save(FINAL_MODEL_PATH)
best_model.save_weights(WEIGHTS_PATH)

with open(WORK_DIR / "aptos_resnet50_dr_label_map.json", "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)

print("Saved full model :", FINAL_MODEL_PATH)
print("Saved weights    :", WEIGHTS_PATH)
print("Saved label map  :", WORK_DIR / "aptos_resnet50_dr_label_map.json")
print()
print("Application model input : RGB fundus image resized/padded to", IMG_SIZE, "x", IMG_SIZE)
print("Application model output: 5 probabilities for grades [0, 1, 2, 3, 4]")
print("Predicted grade          : argmax(probabilities)")

In [ ]:
# ============================================================
# 20. OPTIONAL — APTOS test predictions / submission.csv
# ============================================================

def make_test_dataset(paths):
    ds = tf.data.Dataset.from_tensor_slices(paths.astype(str))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds

if TEST_CSV.exists() and TEST_IMG_DIR.exists():
    test_df = pd.read_csv(TEST_CSV).copy()
    test_df["image_path"] = test_df["id_code"].apply(
        lambda x: resolve_image_path(x, TEST_IMG_DIR)
    )

    if test_df["image_path"].isna().any():
        print("Some test images could not be resolved. Submission skipped.")
    else:
        test_ds = make_test_dataset(test_df["image_path"].values)
        test_probs = best_model.predict(test_ds, verbose=1)
        test_pred = np.argmax(test_probs, axis=1).astype(int)

        if SAMPLE_SUBMISSION.exists():
            submission = pd.read_csv(SAMPLE_SUBMISSION)
            submission["diagnosis"] = test_pred
        else:
            submission = pd.DataFrame({
                "id_code": test_df["id_code"].values,
                "diagnosis": test_pred,
            })

        submission_path = WORK_DIR / "submission.csv"
        submission.to_csv(submission_path, index=False)
        display(submission.head())
        print("Saved:", submission_path)
else:
    print("APTOS test files are not attached. Training/export is unaffected.")

## What to check before accepting the trained model

Do **not** accept it only because validation accuracy is high.

Check:

- **QWK** — important for ordered DR grades.
- **Macro-F1** — protects against ignoring minority grades.
- **Per-class recall** — especially Grades 1, 3 and 4.
- **Confusion matrix** — severe mistakes such as Grade 0 ↔ Grade 4 should be rare.
- **Training/validation curves** — a widening loss gap suggests overfitting.
- **Independent external validation** — required before making clinical-performance claims.

### Main Kaggle output

`/kaggle/working/aptos_resnet50_dr_grading_final.keras`

Other outputs:

- `aptos_resnet50_dr_grading.weights.h5`
- `aptos_resnet50_dr_label_map.json`
- `aptos_resnet50_metrics.json`
- `aptos_resnet50_classification_report.csv`
- `aptos_resnet50_confusion_matrix.png`
- `aptos_resnet50_full_history.csv`
- `submission.csv` if the APTOS test set is attached